In [2]:
import pandas as pd
import numpy as np
from sklearn.neighbors import BallTree

print("1. Loading all three datasets...")
base_df = pd.read_csv('global_bleaching_environmental.csv', low_memory=False)
plastic_df = pd.read_csv('ocean_plastic_pollution_data.csv')
oil_df = pd.read_csv('oil_incidents.csv')



1. Loading all three datasets...


In [3]:
print("2. Cleaning missing coordinates and formatting oil data...")
base_valid = base_df.dropna(subset=['Latitude_Degrees', 'Longitude_Degrees']).copy()
plastic_valid = plastic_df.dropna(subset=['Latitude', 'Longitude']).copy()
oil_valid = oil_df.dropna(subset=['lat', 'lon']).copy()

# Fill missing oil spill volumes with the median spill size (Standard ML Practice)
oil_median_gallons = oil_valid['max_ptl_release_gallons'].median()
oil_valid['max_ptl_release_gallons'] = oil_valid['max_ptl_release_gallons'].fillna(oil_median_gallons)

# Convert all coordinates to radians for the Earth's curvature math
base_rad = np.radians(base_valid[['Latitude_Degrees', 'Longitude_Degrees']].values)
plastic_rad = np.radians(plastic_valid[['Latitude', 'Longitude']].values)
oil_rad = np.radians(oil_valid[['lat', 'lon']].values)

print("3. Spatial Merge 1: Mapping the Plastic Hotspots...")
tree_plastic = BallTree(plastic_rad, metric='haversine')
dist_plastic, idx_plastic = tree_plastic.query(base_rad, k=1)

base_valid['Distance_to_Plastic_km'] = dist_plastic.flatten() * 6371
base_valid['Nearest_Plastic_Weight_kg'] = plastic_valid.iloc[idx_plastic.flatten()]['Plastic_Weight_kg'].values

print("4. Applying the 75th Percentile Scientific Threshold...")
threshold_kg = plastic_valid['Plastic_Weight_kg'].quantile(0.75)
print(f"Calculated 75th Percentile Threshold: {threshold_kg:.2f} kg")

# Create the binary 'Is_Heavily_Polluted' flag
base_valid['Is_Heavily_Polluted'] = ((base_valid['Nearest_Plastic_Weight_kg'] > threshold_kg) & 
                                     (base_valid['Distance_to_Plastic_km'] < 50)).astype(int)


2. Cleaning missing coordinates and formatting oil data...
3. Spatial Merge 1: Mapping the Plastic Hotspots...
4. Applying the 75th Percentile Scientific Threshold...
Calculated 75th Percentile Threshold: 376.71 kg


In [4]:

print("5. Spatial Merge 2: Mapping Historical NOAA Oil Spills...")
tree_oil = BallTree(oil_rad, metric='haversine')
dist_oil, idx_oil = tree_oil.query(base_rad, k=1)

base_valid['Distance_to_Oil_Spill_km'] = dist_oil.flatten() * 6371
base_valid['Nearest_Oil_Spill_Gallons'] = oil_valid.iloc[idx_oil.flatten()]['max_ptl_release_gallons'].values

print("6. Formatting final features and saving...")
base_valid['Percent_Bleaching'] = pd.to_numeric(base_valid['Percent_Bleaching'], errors='coerce')
model_df = base_valid.dropna(subset=['Percent_Bleaching'])

# Define final predictive features including our new Oil and Threshold data
features = ['ClimSST', 'Temperature_Mean', 'Depth_m', 'Windspeed', 'SSTA']
for col in features:
    model_df.loc[:, col] = pd.to_numeric(model_df[col], errors='coerce')
    
model_df = model_df.dropna(subset=features)

cols_to_keep = ['Latitude_Degrees', 'Longitude_Degrees'] + features + \
               ['Distance_to_Plastic_km', 'Nearest_Plastic_Weight_kg', 'Is_Heavily_Polluted',
                'Distance_to_Oil_Spill_km', 'Nearest_Oil_Spill_Gallons', 'Percent_Bleaching']

final_df = model_df[cols_to_keep].copy()

# Save the final robust dataset
output_file = 'dataset_with_oil.csv'
final_df.to_csv(output_file, index=False)

print(f"\nSuccess! Merged multi-stressor dataset saved as: {output_file}")
print(f"Dataset Dimensions: {final_df.shape}")
print("\nPreview of new Anthropogenic Stressor Columns:")
print(final_df[['Is_Heavily_Polluted', 'Distance_to_Oil_Spill_km', 'Nearest_Oil_Spill_Gallons']].head(3).to_string())

5. Spatial Merge 2: Mapping Historical NOAA Oil Spills...
6. Formatting final features and saving...

Success! Merged multi-stressor dataset saved as: dataset_with_oil.csv
Dataset Dimensions: (32716, 13)

Preview of new Anthropogenic Stressor Columns:
   Is_Heavily_Polluted  Distance_to_Oil_Spill_km  Nearest_Oil_Spill_Gallons
0                    0                 30.010717                   200000.0
1                    0               1990.693326                       41.0
2                    0                 11.470838                     3450.0
